# SHIVA-LLM — Objective 02 — Stage 4B
## Simulation-based contextual-bandit policy training

**Colab runtime:** Python 3 / CPU (12–16 GB RAM recommended).  
**Required uploads:** `objective02_stage4a_simulator.zip`, `objective02_stage4ac_reward_redesign.zip`.

**Scientific scope:** One-step contextual bandit with assumed action effects and normative utility. There are **no authenticated logged interventions**, no identified causal effects, and no validated sequential transitions. Thus no actual student learning improvements are inferred. Test labels are never used for training, model selection, or simulator design. The Stage 4A-C reward specification is frozen.

In [ ]:
import io, json, zipfile, pathlib, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import joblib
from google.colab import files

SEED=42
OUT=pathlib.Path('/content/objective02_stage4b'); OUT.mkdir(exist_ok=True)
uploaded=files.upload()
paths=['objective02_stage4a_simulator.zip','objective02_stage4ac_reward_redesign.zip']
assert all(pathlib.Path(p).exists() for p in paths), 'Upload both required ZIPs'
za,zc=map(zipfile.ZipFile,paths)
assert za.testzip() is None and zc.testzip() is None
print('Stage 4A:',za.namelist()); print('Stage 4A-C:',zc.namelist())

In [ ]:
data=pd.read_csv(io.BytesIO(za.read('response_model_predictions.csv')))
report=json.loads(zc.read('stage4ac_report.json'))
assert report['stage']=='4A-C'
assert report['validation_only_selection'] and report['locked_test_audit']
assert report['causal_effects_identified'] is False
assert {'split','user_id','order_id','observed_correct','p_observed'}<=set(data.columns)
assert set(data.split.unique())=={'train','val','test'}
assert not data.duplicated(['user_id','order_id']).any()
assert data.p_observed.between(0,1).all()
ids={s:set(data.loc[data.split==s,'user_id']) for s in ('train','val','test')}
assert not (ids['train']&ids['val'] or ids['train']&ids['test'] or ids['val']&ids['test'])
print(data.groupby('split').agg(events=('user_id','size'),learners=('user_id','nunique')))
DESIGN_NAME=report['selected_design']
G=report['designs'][DESIGN_NAME]
SCENARIOS=report['assumed_effect_scenarios']
CHALLENGE=np.array(report['challenge_credits'],float)
COST=np.array(report['assumed_costs'],float)
ACTIONS=['easy','medium','hard']
assert len(CHALLENGE)==len(COST)==3
print('Frozen reward:',DESIGN_NAME,G,'Scenarios:',SCENARIOS)

### Frozen simulator and reward

For context \(p\) and action \(a\), the simulator assumes \(q_a(p)=\sigma(\operatorname{logit}(p)+\delta_a)\) where \(\delta=(d,0,-d)\). Expected utility is \(w_cq_a+w_mq_ac_a+w_s\exp[-(q_a-	au)^2/(2h^2)]-w_kk_a\). All parameters come from Stage 4A-C; this stage does not change them.

**Training target:** maximize expected *simulated* utility for the moderate scenario. Other scenarios are out-of-assumption stress tests, not extra training data.

In [ ]:
def reward_matrix(p, scenario):
    p=np.clip(np.asarray(p,float),1e-6,1-1e-6)
    d=float(SCENARIOS[scenario])
    l=np.log(p/(1-p))
    q=1/(1+np.exp(-np.clip(l[:,None]+np.array([d,0,-d])[None,:],-35,35)))
    s=np.exp(-.5*((q-G['tau'])/G['h'])**2)
    u=G['wc']*q+G['wm']*q*CHALLENGE[None,:]+G['ws']*s-G['wk']*COST[None,:]
    return q,u

def x_features(df):
    # p_observed is a pre-response prediction; no true label, future response or user identifier.
    return df[['p_observed']].to_numpy(dtype=np.float64)

splits={s:data[data.split==s].reset_index(drop=True) for s in ('train','val','test')}
Xtr=x_features(splits['train']); Xval=x_features(splits['val'])
_,Rtr=reward_matrix(Xtr[:,0],'moderate')
_,Rval=reward_matrix(Xval[:,0],'moderate')
ytr=Rtr.argmax(axis=1)
print('Train simulated oracle-action distribution:',dict(zip(ACTIONS,np.bincount(ytr,minlength=3).tolist())))

### Train candidates and select only on validation

Candidate policies: decision trees (depth 2, 3, 5, 8) and multinomial logistic regression. The selection criterion is **mean simulated validation utility under the moderate assumed-effect scenario**, not observed student outcomes. Fixed/random/rule-based baselines are evaluated separately.

In [ ]:
candidates={}
for depth in [2,3,5,8]:
    model=DecisionTreeClassifier(max_depth=depth,min_samples_leaf=200,random_state=SEED)
    model.fit(Xtr,ytr)
    candidates[f'tree_depth_{depth}']=model
logreg=make_pipeline(StandardScaler(),LogisticRegression(max_iter=300,random_state=SEED))
logreg.fit(Xtr,ytr)
candidates['logistic_regression']=logreg

selection=[]
for name,model in candidates.items():
    a=model.predict(Xval).astype(int)
    selection.append({'model':name,'val_mean_simulated_reward':float(Rval[np.arange(len(a)),a].mean()),
                      'val_simulated_regret':float((Rval.max(axis=1)-Rval[np.arange(len(a)),a]).mean()),
                      'val_action_diversity':float(1-np.bincount(a,minlength=3).max()/len(a))})
selection=pd.DataFrame(selection).sort_values(['val_mean_simulated_reward','model'],ascending=[False,True]).reset_index(drop=True)
SELECTED=str(selection.iloc[0]['model']); POLICY=candidates[SELECTED]
selection.to_csv(OUT/'validation_model_selection.csv',index=False)
display(selection)
print('Frozen selected model:',SELECTED)

In [ ]:
# Baselines defined independently of held-out outcomes.
def actions_for(name,df,model=None,seed=SEED):
    p=df.p_observed.to_numpy()
    if name=='learned': return model.predict(x_features(df)).astype(int)
    if name=='fixed_easy': return np.zeros(len(df),dtype=int)
    if name=='fixed_medium': return np.ones(len(df),dtype=int)
    if name=='fixed_hard': return np.full(len(df),2,dtype=int)
    if name=='random': return np.random.default_rng(seed).integers(0,3,size=len(df))
    if name=='readiness_rule': return np.where(p<.4,0,np.where(p<.7,1,2)).astype(int)
    raise ValueError(name)

POLICIES=['learned','fixed_easy','fixed_medium','fixed_hard','random','readiness_rule']
def evaluate(df,scenario,model,seed=SEED):
    q,r=reward_matrix(df.p_observed.to_numpy(),scenario)
    oracle=r.max(axis=1)
    records=[]
    for name in POLICIES:
        a=actions_for(name,df,model,seed)
        chosen=r[np.arange(len(a)),a]
        row={'scenario':scenario,'policy':name,'events':len(df),'learners':df.user_id.nunique(),
             'mean_simulated_reward':float(chosen.mean()),'mean_simulated_regret':float((oracle-chosen).mean()),
             'mean_assumed_correctness':float(q[np.arange(len(a)),a].mean()),
             'action_diversity':float(1-np.bincount(a,minlength=3).max()/len(a))}
        row.update({f'fraction_{label}':float((a==i).mean()) for i,label in enumerate(ACTIONS)})
        records.append(row)
    return pd.DataFrame(records)

# Validation baseline diagnostics (not used to alter reward parameters).
val_baselines=pd.concat([evaluate(splits['val'],s,POLICY) for s in SCENARIOS],ignore_index=True)
val_baselines.to_csv(OUT/'validation_policy_comparison.csv',index=False)
display(val_baselines[val_baselines.scenario=='moderate'])

### Locked test evaluation

The policy is now frozen. Report the test results for all four effect scenarios, including zero effect. This is **in-simulator policy evaluation**, not offline policy evaluation from real intervention logs.

In [ ]:
test=splits['test']
test_results=pd.concat([evaluate(test,s,POLICY) for s in SCENARIOS],ignore_index=True)
test_results.to_csv(OUT/'locked_test_policy_comparison.csv',index=False)
display(test_results)

fig,ax=plt.subplots(figsize=(9,5))
for name in POLICIES:
    d=test_results[test_results.policy==name]
    ax.plot(d.scenario,d.mean_simulated_reward,marker='o',label=name)
ax.set(xlabel='Assumed action-effect scenario',ylabel='Mean assumed utility',title='Locked-test simulated policy comparison')
ax.legend(ncol=2,fontsize=8);fig.tight_layout()
fig.savefig(OUT/'test_policy_rewards.png',dpi=170)
plt.show()

### Learner-cluster bootstrap

Resample test learners, keeping all their events together. The resulting confidence intervals describe **sampling variability conditional on the assumed simulator**. They do not capture uncertainty about real action effects. The comparison is learned policy minus the strongest fixed-action baseline *within each bootstrap sample*.

In [ ]:
def cluster_bootstrap(df,scenario,model,n_boot=300,seed=42):
    p=df.p_observed.to_numpy(); _,r=reward_matrix(p,scenario)
    learned=actions_for('learned',df,model)
    learned_u=r[np.arange(len(df)),learned]
    groups=df.groupby('user_id').indices
    keys=np.array(list(groups.keys()))
    gen=np.random.default_rng(seed)
    diffs=[]
    for _ in range(n_boot):
        sample=gen.choice(keys,size=len(keys),replace=True)
        ix=np.concatenate([groups[k] for k in sample])
        diff=learned_u[ix].mean()-r[ix].mean(axis=0).max()
        diffs.append(float(diff))
    return {'scenario':scenario,'bootstrap_reps':n_boot,
            'learned_minus_best_fixed_mean':float(np.mean(diffs)),
            'ci_95_low':float(np.quantile(diffs,.025)),
            'ci_95_high':float(np.quantile(diffs,.975))}
bootstrap=pd.DataFrame([cluster_bootstrap(test,s,POLICY) for s in SCENARIOS])
bootstrap.to_csv(OUT/'learner_bootstrap_policy_comparison.csv',index=False)
display(bootstrap)

In [ ]:
# Readiness-group action distribution for the frozen learned policy.
act=actions_for('learned',test,POLICY)
readiness=pd.cut(test.p_observed,[0,.4,.7,1.000001],labels=['low','medium','high'],right=False,include_lowest=True)
readiness_rows=[]
for label in ['low','medium','high']:
    mask=(readiness==label).to_numpy()
    if mask.sum()==0:continue
    readiness_rows.append({'readiness':label,'events':int(mask.sum()),
                           **{f'fraction_{a}':float((act[mask]==i).mean()) for i,a in enumerate(ACTIONS)}})
readiness_df=pd.DataFrame(readiness_rows)
readiness_df.to_csv(OUT/'learned_action_by_readiness.csv',index=False)
display(readiness_df)

In [ ]:
# Save model, frozen assumptions, reproducibility and limitations.
joblib.dump(POLICY,OUT/'stage4b_selected_policy.joblib')
summary={
 'stage':'4B','seed':SEED,'selected_policy':SELECTED,'training_scenario':'moderate',
 'frozen_stage4ac_reward':DESIGN_NAME,'reward_parameters':G,
 'effect_scenarios':SCENARIOS,'challenge_credits':CHALLENGE.tolist(),'assumed_costs':COST.tolist(),
 'train_events':len(splits['train']),'val_events':len(splits['val']),'test_events':len(splits['test']),
 'train_only_fit':True,'validation_only_model_selection':True,'test_not_used_for_selection':True,
 'simulated_action_effects_only':True,'authentic_logged_actions':False,
 'causal_instructional_improvement_established':False,'sequential_RL_environment_validated':False,
 'notes':'Policy learns a one-step mapping from pre-response correctness prediction to an assumed reward-maximizing difficulty action.'}
(OUT/'stage4b_report.json').write_text(json.dumps(summary,indent=2))
(OUT/'README.txt').write_text('Stage 4B simulation-only contextual-bandit policy optimization.\n'
 'Do not interpret simulated rewards as actual learning gains or causal effects.\n'
 'Inputs: Stage 4A and Stage 4A-C ZIP archives.\n')
archive=pathlib.Path('/content/objective02_stage4b_policy_optimization.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file():z.write(f,arcname=f.name)
print('Output:',archive,'bytes:',archive.stat().st_size)
files.download(str(archive))